# Análisis bursátil chileno: rentabilidad y riesgo (2015–2026)

Pipeline completo: descarga de precios con `yfinance` → SQLite → vistas y consultas SQL → CSV para dashboards.

**Alcance:** 5 acciones chilenas (Falabella, SQM-B, Banco de Chile, Cencosud, Enel Américas), datos diarios desde 2015-01-02.

> **Nota sobre `^IPSA`:** Yahoo Finance no devolvió datos para el índice (`YFTzMissingError`), por lo que el análisis **no incluye benchmark**. La tabla `empresas` lo lista pero no tiene precios asociados.

> **Ejecución:** pensado para Google Colab (usa `google.colab.files`). Fuera de Colab, comentar las líneas `files.download(...)`.

**Convenciones:** retornos sobre `Adj Close`; anualización con 252 días hábiles.


## 1. Descarga y limpieza de datos

In [10]:
# 1. Instalar la librería financiera
!pip install yfinance

# 2. Importar las herramientas
import yfinance as yf
import pandas as pd
from google.colab import files

# 3. Definir los tickers (IPSA y 5 empresas chilenas de ejemplo)
# Nota: ^IPSA a veces falla por zona horaria en la API, por lo que agregamos manejo de errores robusto.
tickers = ["^IPSA", "FALABELLA.SN", "SQM-B.SN", "CHILE.SN", "CENCOSUD.SN", "ENELAM.SN"]
datos = []

# 4. Descargar los datos
for t in tickers:
    print(f"Descargando {t}...")
    try:
        # Forzamos auto_adjust=False para mantener 'Adj Close'
        df = yf.download(t, start="2015-01-01", end="2026-10-31", auto_adjust=False)

        if not df.empty:
            df = df.copy()
            df.reset_index(inplace=True)

            # Limpiar el formato de las columnas MultiIndex si existen
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['ticker'] = t
            datos.append(df)
    except Exception as e:
        print(f"Error al descargar {t}: {e}")

if datos:
    # 5. Unir y limpiar
    df_final = pd.concat(datos, ignore_index=True)
    df_final = df_final.rename(columns={
        "Date": "fecha",
        "Open": "apertura",
        "High": "maximo",
        "Low": "minimo",
        "Close": "cierre",
        "Adj Close": "cierre_ajustado",
        "Volume": "volumen"
    })

    # Asegurar que todas las columnas requeridas existen en el df final
    columnas_requeridas = ['ticker', 'fecha', 'apertura', 'maximo', 'minimo', 'cierre', 'cierre_ajustado', 'volumen']
    df_final = df_final[[col for col in columnas_requeridas if col in df_final.columns]]

    # 6. Guardar y descargar a tu PC
    df_final.to_csv("bolsa_chile_crudo.csv", index=False)
    print("Generando archivo y preparando descarga...")
    display(df_final.head())
    files.download("bolsa_chile_crudo.csv")
else:
    print("No se descargaron datos válidos para procesar.")

[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^IPSA']: YFTzMissingError('possibly delisted; no timezone found')
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Descargando ^IPSA...
Descargando FALABELLA.SN...
Descargando SQM-B.SN...
Descargando CHILE.SN...
Descargando CENCOSUD.SN...



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Descargando ENELAM.SN...
Generando archivo y preparando descarga...


Price,ticker,fecha,apertura,maximo,minimo,cierre,cierre_ajustado,volumen
0,FALABELLA.SN,2015-01-02,4071.399902,4100.000000,4060.000000,4083.500000,3404.407227,224842
1,FALABELLA.SN,2015-01-05,4083.500000,4083.500000,3981.000000,4007.300049,3340.879883,810990
2,FALABELLA.SN,2015-01-06,4007.300049,4020.000000,3960.000000,3973.500000,3312.700684,957030
3,FALABELLA.SN,2015-01-07,3973.500000,4020.000000,3932.300049,3945.199951,3289.107422,571950
4,FALABELLA.SN,2015-01-08,3945.199951,4000.100098,3945.000000,3979.100098,3317.369385,677920


In [3]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14730 entries, 0 to 14729
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ticker           14730 non-null  object        
 1   fecha            14730 non-null  datetime64[ns]
 2   apertura         14730 non-null  float64       
 3   maximo           14730 non-null  float64       
 4   minimo           14730 non-null  float64       
 5   cierre           14730 non-null  float64       
 6   cierre_ajustado  14730 non-null  float64       
 7   volumen          14730 non-null  int64         
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 920.8+ KB


In [4]:
# Muestra balanceada (200 primeras filas por ticker) para versionar en el repo
muestra_precios = df_final.groupby('ticker').head(200)
muestra_precios.to_csv("muestra_precios.csv", index=False)
files.download("muestra_precios.csv")

## 2. Carga en SQLite

In [5]:
import sqlite3

# 1. Crear conexión a una base de datos local
conexion = sqlite3.connect("bolsa_chile.db")

# 2. Subir el DataFrame a una tabla SQL llamada 'precios_diarios'
df_final.to_sql("precios_diarios", conexion, if_exists="replace", index=False)

print("Tabla SQL 'precios_diarios' creada con éxito.")

Tabla SQL 'precios_diarios' creada con éxito.


In [6]:
consulta_prueba = """
SELECT COUNT(*) AS total_filas
FROM precios_diarios;
"""
resultado = pd.read_sql_query(consulta_prueba, conexion)
display(resultado)

,total_filas
0,14730


## 3. Exploración: volúmenes y precios

In [11]:
consulta_top_10 = """
-- ESCRIBE TU SQL AQUÍ ABAJO:
SELECT ticker, fecha, apertura, maximo, minimo, cierre, volumen
FROM precios_diarios
WHERE volumen IS NOT NULL
ORDER BY volumen DESC
LIMIT 10;
"""

# Ejecutamos la consulta y mostramos el resultado
resultado_top_10 = pd.read_sql_query(consulta_top_10, conexion)
display(resultado_top_10)

,ticker,fecha,apertura,maximo,minimo,cierre,volumen
0,ENELAM.SN,2026-02-27 00:00:00,78.000000,79.449997,76.790001,78.500000,3391175213
1,CHILE.SN,2019-08-27 00:00:00,101.860001,101.860001,99.550003,101.860001,1400803755
2,CHILE.SN,2024-10-03 00:00:00,111.199997,112.459999,109.790001,112.449997,897370055
3,CHILE.SN,2024-09-30 00:00:00,114.199997,114.699997,111.989998,114.699997,872077097
4,ENELAM.SN,2017-07-05 00:00:00,127.954552,128.840393,124.893478,128.486053,830170469
5,ENELAM.SN,2021-04-20 00:00:00,113.000000,113.000000,106.000000,113.000000,777624400
6,CHILE.SN,2017-07-05 00:00:00,81.310669,83.780273,81.077682,82.773788,712903369
7,CHILE.SN,2020-11-30 00:00:00,67.150002,68.330002,66.180000,67.150002,681447889
8,CHILE.SN,2017-04-05 00:00:00,75.392929,76.520561,75.020157,76.418045,545874032
9,CHILE.SN,2019-09-17 00:00:00,103.800003,104.900002,103.160004,103.800003,523607520


In [13]:
consulta_volumen_promedio = """
-- Consulta para obtener el volumen promedio por ticker ordenado de mayor a menor
SELECT
    ticker,
    AVG(volumen) AS volumen_promedio
FROM precios_diarios
GROUP BY ticker
ORDER BY volumen_promedio DESC;
"""

# Ejecutar la consulta y mostrar el resultado
resultado_promedio = pd.read_sql_query(consulta_volumen_promedio, conexion)
display(resultado_promedio)

,ticker,volumen_promedio
0,CHILE.SN,3.665077e+07
1,ENELAM.SN,2.970763e+07
2,CENCOSUD.SN,1.812424e+06
3,FALABELLA.SN,1.137165e+06
4,SQM-B.SN,1.846670e+05


In [14]:
consulta_falabella_2020 = """
-- Consulta para obtener ticker, fecha y cierre de FALABELLA.SN en el año 2020
SELECT ticker, fecha, cierre
FROM precios_diarios
WHERE ticker = 'FALABELLA.SN'
  AND fecha >= '2020-01-01'
  AND fecha <= '2020-12-31'
ORDER BY fecha;
"""

# Ejecutar la consulta y mostrar el resultado
resultado_falabella = pd.read_sql_query(consulta_falabella_2020, conexion)
display(resultado_falabella)

,ticker,fecha,cierre
0,FALABELLA.SN,2020-01-02 00:00:00,3350.000000
1,FALABELLA.SN,2020-01-03 00:00:00,3350.000000
2,FALABELLA.SN,2020-01-06 00:00:00,3390.000000
3,FALABELLA.SN,2020-01-07 00:00:00,3450.000000
4,FALABELLA.SN,2020-01-08 00:00:00,3455.899902
...,...,...,...
246,FALABELLA.SN,2020-12-23 00:00:00,2531.000000
247,FALABELLA.SN,2020-12-24 00:00:00,2550.000000
248,FALABELLA.SN,2020-12-28 00:00:00,2584.000000
249,FALABELLA.SN,2020-12-29 00:00:00,2639.000000


## 4. Retornos diarios y anuales (vista `v_retornos`)

In [15]:
# Primero borramos la vista si existe para que no de error al re-ejecutar
conexion.execute("DROP VIEW IF EXISTS v_retornos;")

consulta_vista = """
CREATE VIEW v_retornos AS
SELECT
    ticker,
    fecha,
    cierre_ajustado,
    (cierre_ajustado / LAG(cierre_ajustado) OVER (PARTITION BY ticker ORDER BY fecha)) - 1 AS ret_diario
FROM precios_diarios;
"""

# Ejecutamos la creación de la vista
conexion.execute(consulta_vista)

# Comprobamos que funcionó viendo los primeros datos de Falabella
comprobacion = pd.read_sql_query("SELECT * FROM v_retornos WHERE ticker = 'FALABELLA.SN' LIMIT 5;", conexion)
display(comprobacion)

,ticker,fecha,cierre_ajustado,ret_diario
0,FALABELLA.SN,2015-01-02 00:00:00,3404.407227,NaN
1,FALABELLA.SN,2015-01-05 00:00:00,3340.879883,-0.018660
2,FALABELLA.SN,2015-01-06 00:00:00,3312.700684,-0.008435
3,FALABELLA.SN,2015-01-07 00:00:00,3289.107422,-0.007122
4,FALABELLA.SN,2015-01-08 00:00:00,3317.369385,0.008593


In [16]:
consulta_anual = """
SELECT
    ticker,
    STRFTIME('%Y', fecha) AS anio,
    AVG(ret_diario) * 252 AS retorno_anual_aprox
FROM v_retornos
WHERE ret_diario IS NOT NULL
GROUP BY ticker, anio
ORDER BY ticker, anio;
"""

resultado_anual = pd.read_sql_query(consulta_anual, conexion)
display(resultado_anual.head(10))

,ticker,anio,retorno_anual_aprox
0,CENCOSUD.SN,2015,-0.000044
1,CENCOSUD.SN,2016,0.355619
2,CENCOSUD.SN,2017,0.028920
3,CENCOSUD.SN,2018,-0.330296
4,CENCOSUD.SN,2019,-0.183871
5,CENCOSUD.SN,2020,0.393328
6,CENCOSUD.SN,2021,0.379125
7,CENCOSUD.SN,2022,0.109879
8,CENCOSUD.SN,2023,0.269158
9,CENCOSUD.SN,2024,0.332301


## 5. Sectores, riesgo/retorno y exportación para el dashboard

La volatilidad es la **desviación estándar** de los retornos diarios por √252. (Versión previa del notebook reportaba la varianza anualizada; corregido.)

In [17]:
# 1. Crear el DataFrame de sectores
datos_empresas = {
    'ticker': ['^IPSA', 'FALABELLA.SN', 'SQM-B.SN', 'CHILE.SN', 'CENCOSUD.SN', 'ENELAM.SN'],
    'nombre': ['Índice IPSA', 'Falabella', 'SQM', 'Banco de Chile', 'Cencosud', 'Enel Américas'],
    'sector': ['Índice', 'Retail', 'Minería', 'Banca', 'Retail', 'Energía']
}
df_empresas = pd.DataFrame(datos_empresas)

# 2. Subirlo a SQL
df_empresas.to_sql("empresas", conexion, if_exists="replace", index=False)
print("Tabla 'empresas' creada y subida con éxito.")

Tabla 'empresas' creada y subida con éxito.


In [18]:
# SQLite no trae SQRT ni STDDEV en todas las builds: registramos SQRT desde Python
import math
conexion.create_function("SQRT", 1, math.sqrt)

consulta_joins = """
SELECT
    e.sector,
    r.ticker,
    AVG(r.ret_diario) * 252 AS retorno_anual,
    -- Volatilidad anualizada = desviación estándar diaria * raíz(252)
    SQRT(SUM((r.ret_diario - promedios.media) * (r.ret_diario - promedios.media)) / (COUNT(r.ret_diario) - 1) * 252) AS volatilidad_anual
FROM v_retornos r
JOIN empresas e ON r.ticker = e.ticker
JOIN (SELECT ticker, AVG(ret_diario) AS media FROM v_retornos GROUP BY ticker) promedios ON r.ticker = promedios.ticker
WHERE r.ret_diario IS NOT NULL
GROUP BY e.sector, r.ticker
ORDER BY e.sector, retorno_anual DESC;
"""

resultado_joins = pd.read_sql_query(consulta_joins, conexion)
display(resultado_joins)

    sector        ticker  retorno_anual  volatilidad_anual
0  Banca     CHILE.SN          0.168728           0.234171
1  Energía   ENELAM.SN         0.013767           0.259828
2  Minería   SQM-B.SN          0.211979           0.411248
3  Retail    CENCOSUD.SN       0.105947           0.296714
4  Retail    FALABELLA.SN      0.095831           0.293615


In [19]:
consulta_dashboard = """
SELECT
    r.fecha,
    e.sector,
    r.ticker,
    e.nombre,
    r.cierre_ajustado,
    r.ret_diario
FROM v_retornos r
JOIN empresas e ON r.ticker = e.ticker
"""

# Extraer los datos y guardar el CSV final
df_dashboard = pd.read_sql_query(consulta_dashboard, conexion)
df_dashboard.to_csv("datos_dashboard.csv", index=False)

# Descargar a tu computador
from google.colab import files
files.download("datos_dashboard.csv")
print("Archivo 'datos_dashboard.csv' listo para Looker Studio.")

Archivo 'datos_dashboard.csv' listo para Looker Studio.


## 6. Conclusiones

| Sector | Ticker | Retorno anual (aprox.) | Volatilidad anual |
|---|---|---|---|
| Banca | CHILE.SN | 16.9% | 23.4% |
| Energía | ENELAM.SN | 1.4% | 26.0% |
| Minería | SQM-B.SN | 21.2% | 41.1% |
| Retail | CENCOSUD.SN | 10.6% | 29.7% |
| Retail | FALABELLA.SN | 9.6% | 29.4% |

- SQM-B tiene el mayor retorno (~21%) pero también la mayor volatilidad (~41%).
- Banco de Chile ofrece el mejor equilibrio riesgo/retorno de la muestra (~17% con ~23%).
- Enel Américas rindió menos de 2% anual con ~26% de volatilidad.

**Limitaciones:** 5 empresas, sin benchmark (IPSA), retorno anual = media diaria × 252 (aproximación aritmética, no CAGR), sin tasa libre de riesgo ni Sharpe, rentabilidades pasadas no garantizan resultados futuros. No es asesoría de inversión.
